## step_log_test2
Exercises the audit-logging bracket end to end with a **realistic Bronze no-files
early-exit** and validates exit-early-AND-continue plus error logging. The job runs this
notebook twice against real Volume folders:

- **scenario=no_files**: `dbutils.fs.ls(SOURCE_PATH)` returns no CSVs → writes a
  `NO_FILES` step row → `dbutils.notebook.exit(...)`. The exit is DEFERRED to *after* the
  try (the no-files CHECK stays inside it), so it is never swallowed by `except Exception`;
  the task **succeeds**, so the job continues.
- **scenario=has_files**: the folder contains a CSV → the notebook proceeds, runs the
  divide cell, and closes `SUCCEEDED`. It depends on the no_files task, so the fact that it
  runs proves the early exit let the job continue.

**To test the FAILED path:** edit the divide cell from `100 / 10` to `100 / 0`. The
has_files task then raises → `step.fail(e)` writes `STATUS_FAILED` → re-raises → the task
fails → `finalize` (run_if ALL_DONE) still runs and marks the run `failed`. (A bad
SOURCE_PATH would likewise be caught inside the try and logged via `step.fail`.)

Why the exit is outside the try: `dbutils.notebook.exit()` raises an exception that
`except Exception` would catch and swallow (Databricks community + MS Q&A). Calling it
where no `except` can intercept it is the robust pattern — there is no reliable
`dbutils.NotebookExit` class to guard on.

Precondition: the raw Volume + audit tables must exist (run `catalog_setup` first).

In [0]:
%run "../libs/notebook_init"

In [0]:
# Parameters + constants + test scaffolding.
# notebook_init injected Utils, StepLog, AUDIT, RAW_FILES, PIPELINE_RUN_ID, STATUS_*.
dbutils.widgets.text("scenario", "has_files")      # "no_files" | "has_files"
dbutils.widgets.text("step_sequence", "1")
SCENARIO      = dbutils.widgets.get("scenario").strip().lower()
STEP_SEQUENCE = 2

# Real Volume folder for this scenario. A real Bronze notebook reads a fixed SOURCE_PATH;
# here it varies by scenario so one notebook can exercise both branches.
SOURCE_PATH = f"{RAW_FILES}step_log_test/{SCENARIO}/"

# --- test scaffolding (NOT part of the real pattern) ---------------------------------
# Stand in for download_sources having (or not having) landed files. A real Bronze
# notebook does NOT create files — it only reads SOURCE_PATH. Done before opening the
# StepLog row so a scaffolding failure doesn't leave an orphan RUNNING row.

print(f"step_log_test: scenario={SCENARIO} source_path={SOURCE_PATH}")

In [0]:
# Open the pipeline_step_log row (RUNNING).
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "bronze",
    target_table    = None,
)
print(f"step_log_test: step_log_id={step.step_log_id}")

In [0]:
# --- file validation + no-files early exit (the REAL Bronze pattern) -----------------
# The no-files CHECK runs INSIDE the try, so a failed dbutils.fs.ls (e.g. a bad path) is
# logged via step.fail. Only the dbutils.notebook.exit() call is DEFERRED to AFTER the
# try — because exit() raises an exception that `except Exception` would otherwise swallow.
no_files = False
try:
   
    files = ["testing.csv"]
    if not files:
        no_files = True
    else:
        step.rows_read = len(files)
        print(f"step_log_test: found {len(files)} CSV file(s)")
except Exception as e:
    step.fail(e); raise

# OUTSIDE the try: the exit signal is never offered to an except clause → clean exit,
# task SUCCEEDS, job continues to downstream tasks.
if no_files:
    step.no_files()
    dbutils.notebook.exit(f"step_log_test: no CSV files at {SOURCE_PATH} — clean NO_FILES exit")

In [0]:
# --- divide cell ---------------------------------------------------------------------
# >>> Edit the divisor to 0 to manually exercise the FAILED path. <<<
# 100 / 10 = 10.0 (success);  100 / 0 raises -> step.fail -> STATUS_FAILED.
try:
    testvar = 100 / 10
    step.rows_written = int(testvar)
    print(f"step_log_test: testvar = {testvar}")
except Exception as e:
    step.fail(e); raise

In [0]:
# --- close the step SUCCEEDED --------------------------------------------------------
try:
    print(f"step_log_test: closing SUCCEEDED (rows_read={step.rows_read}, rows_written={step.rows_written})")
    step.succeed()
except Exception as e:
    step.fail(e); raise